In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    roc_auc_score
)


In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_path = (
    project_root / "data" / "processed"
    / "customer_features.parquet"
)

df = pd.read_parquet(data_path)

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (5249, 8)


,customer_id,last_purchase,frequency,monetary,product_diversity,recency_days,avg_basket_value,purchased_next_30d
0,12346,2011-01-18 10:01:00,12,77556.46,27,225.582639,6463.038333,0
1,12347,2011-08-02 08:48:00,6,4114.18,107,29.633333,685.696667,0
2,12348,2011-04-05 10:47:00,4,1709.40,25,148.550694,427.350000,1
3,12349,2010-10-28 08:23:00,3,2671.14,90,307.650694,890.380000,0
4,12350,2011-02-02 16:01:00,1,334.40,17,210.332639,334.400000,0


In [3]:
feature_columns = [
    "recency_days",
    "frequency",
    "monetary",
    "product_diversity",
    "avg_basket_value"
]

X = df[feature_columns]
y = df["purchased_next_30d"]

print("Features:", X.shape)
print("Target:", y.shape)
print("Missing values:", X.isna().sum().sum())


Features: (5249, 5)
Target: (5249,)
Missing values: 0


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training customers:", len(X_train))
print("Testing customers:", len(X_test))


Training customers: 4199
Testing customers: 1050


In [5]:
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)

logistic_probabilities = (
    logistic_model.predict_proba(X_test)[:, 1]
)

print("LOGISTIC REGRESSION")
print(classification_report(
    y_test,
    logistic_predictions,
    zero_division=0
))

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, logistic_probabilities), 3)
)


LOGISTIC REGRESSION
              precision    recall  f1-score   support

           0       0.83      0.97      0.90       835
           1       0.68      0.25      0.37       215

    accuracy                           0.82      1050
   macro avg       0.76      0.61      0.63      1050
weighted avg       0.80      0.82      0.79      1050

ROC-AUC: 0.789


In [6]:
forest_model = RandomForestClassifier(
    n_estimators=100,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

forest_model.fit(X_train, y_train)

forest_predictions = forest_model.predict(X_test)

forest_probabilities = (
    forest_model.predict_proba(X_test)[:, 1]
)

print("RANDOM FOREST")
print(classification_report(
    y_test,
    forest_predictions,
    zero_division=0
))

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, forest_probabilities), 3)
)


RANDOM FOREST
              precision    recall  f1-score   support

           0       0.83      0.95      0.89       835
           1       0.57      0.27      0.37       215

    accuracy                           0.81      1050
   macro avg       0.70      0.61      0.63      1050
weighted avg       0.78      0.81      0.78      1050

ROC-AUC: 0.773


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, logistic_predictions),
        accuracy_score(y_test, forest_predictions)
    ],
    "Precision": [
        precision_score(y_test, logistic_predictions, zero_division=0),
        precision_score(y_test, forest_predictions, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, logistic_predictions, zero_division=0),
        recall_score(y_test, forest_predictions, zero_division=0)
    ],
    "F1": [
        f1_score(y_test, logistic_predictions, zero_division=0),
        f1_score(y_test, forest_predictions, zero_division=0)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, logistic_probabilities),
        roc_auc_score(y_test, forest_probabilities)
    ]
})

display(results.round(3))


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Logistic Regression,0.823,0.684,0.251,0.367,0.789
1,Random Forest,0.810,0.574,0.270,0.367,0.773


In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

print("Baseline accuracy:",
      round(accuracy_score(y_test, baseline_predictions), 3))

print("Logistic Regression accuracy:",
      round(accuracy_score(y_test, logistic_predictions), 3))

print("Baseline recall:",
      recall_score(y_test, baseline_predictions, zero_division=0))


Baseline accuracy: 0.795
Logistic Regression accuracy: 0.823
Baseline recall: 0.0
